In [0]:
%pip install groq chromadb sentence-transformers -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Pull PTSD trials from ClinicalTrials.gov

In [0]:
import requests
import pandas as pd
import time

def fetch_ptsd_trials(max_studies=3000):
    all_studies = []
    page_token = None

    while len(all_studies) < max_studies:
        params = {
            "format": "json",
            "pageSize": 1000,
            "query.cond": "PTSD",
        }
        if page_token:
            params["pageToken"] = page_token

        resp = requests.get(
            "https://clinicaltrials.gov/api/v2/studies",
            params=params,
            timeout=30
        )
        resp.raise_for_status()
        data = resp.json()
        studies = data.get("studies", [])
        if not studies:
            break

        all_studies.extend(studies)
        print(f"fetched {len(all_studies)} studies...")

        page_token = data.get("nextPageToken")
        if not page_token:
            break
        time.sleep(0.3)

    return all_studies

raw = fetch_ptsd_trials()
print(f"done — {len(raw)} studies")

fetched 1000 studies...
fetched 2000 studies...
fetched 2599 studies...
done — 2599 studies


Parse into structured + unstructured

In [0]:
structured_rows = []
unstructured_rows = []

for study in raw:
    ps           = study.get("protocolSection", {})
    id_mod       = ps.get("identificationModule", {})
    status_mod   = ps.get("statusModule", {})
    design_mod   = ps.get("designModule", {})
    desc_mod     = ps.get("descriptionModule", {})
    elig_mod     = ps.get("eligibilityModule", {})
    sponsor_mod  = ps.get("sponsorCollaboratorsModule", {})
    conditions   = ps.get("conditionsModule", {})
    interventions= ps.get("armsInterventionsModule", {})

    nct_id = id_mod.get("nctId", "")

    # flat fields go to SQL table
    structured_rows.append({
        "nct_id":          nct_id,
        "title":           id_mod.get("briefTitle", ""),
        "status":          status_mod.get("overallStatus", ""),
        "phase":           ", ".join(design_mod.get("phases", [])),
        "study_type":      design_mod.get("studyType", ""),
        "condition":       ", ".join(conditions.get("conditions", [])),
        "intervention":    ", ".join([i.get("name","") for i in interventions.get("interventions", [])]),
        "sponsor":         sponsor_mod.get("leadSponsor", {}).get("name", ""),
        "sponsor_type":    sponsor_mod.get("leadSponsor", {}).get("class", ""),
        "enrollment":      design_mod.get("enrollmentInfo", {}).get("count", None),
        "start_date":      status_mod.get("startDateStruct", {}).get("date", ""),
        "completion_date": status_mod.get("primaryCompletionDateStruct", {}).get("date", ""),
        "min_age":         elig_mod.get("minimumAge", ""),
        "max_age":         elig_mod.get("maximumAge", ""),
        "sex":             elig_mod.get("sex", ""),
    })

    # long text fields go to RAG
    for text_type, text in [
        ("brief_summary",        desc_mod.get("briefSummary", "").strip()),
        ("detailed_description", desc_mod.get("detailedDescription", "").strip()),
        ("eligibility_criteria", elig_mod.get("eligibilityCriteria", "").strip()),
    ]:
        if text:
            unstructured_rows.append({
                "nct_id":    nct_id,
                "text_type": text_type,
                "text":      text,
            })

structured_df   = pd.DataFrame(structured_rows)
unstructured_df = pd.DataFrame(unstructured_rows)

print(f"structured:   {len(structured_df)} rows")
print(f"unstructured: {len(unstructured_df)} rows")

structured:   2599 rows
unstructured: 7138 rows


ETL: clean + chunk + save to Delta (Bronze → Silver)

In [0]:
import re
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Bronze: save raw structured to Delta
string_cols = structured_df.select_dtypes(include=["object"]).columns
structured_df[string_cols] = structured_df[string_cols].fillna("")
structured_df["enrollment"] = pd.to_numeric(structured_df["enrollment"], errors="coerce")

spark.createDataFrame(structured_df).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("ptsd_trials_structured")
print("bronze: ptsd_trials_structured saved")

# Silver: clean text
def clean_text(text):
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s.,;:()\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

unstructured_df["text_clean"] = unstructured_df["text"].apply(clean_text)
unstructured_df = unstructured_df[
    unstructured_df["text_clean"].str.len() >= 50
].reset_index(drop=True)

# Silver: chunk long text into 500-char pieces for embedding
def chunk_text(text, chunk_size=500, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if len(chunk) >= 100:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunked_rows = []
for _, row in unstructured_df.iterrows():
    for i, chunk in enumerate(chunk_text(row["text_clean"])):
        chunked_rows.append({
            "nct_id":      row["nct_id"],
            "text_type":   row["text_type"],
            "chunk_id":    f"{row['nct_id']}_{row['text_type']}_{i}",
            "chunk_text":  chunk,
            "chunk_index": i,
        })

chunked_df = pd.DataFrame(chunked_rows)

spark.createDataFrame(chunked_df).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("ptsd_trials_silver")

print(f"silver: ptsd_trials_silver saved — {len(chunked_df)} chunks")
spark.sql("SELECT text_type, COUNT(*) as chunks FROM ptsd_trials_silver GROUP BY text_type").show()

bronze: ptsd_trials_structured saved
silver: ptsd_trials_silver saved — 25384 chunks
+--------------------+------+
|           text_type|chunks|
+--------------------+------+
|detailed_description| 12162|
|       brief_summary|  5806|
|eligibility_criteria|  7416|
+--------------------+------+



Embedding: Silver → ChromaDB (Gold layer)

In [0]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

# load cleaned chunks from delta
chunked_df = spark.table("ptsd_trials_silver").toPandas()
print(f"loaded {len(chunked_df)} chunks from silver layer")

# MiniLM converts each text chunk into 384 numbers (a vector)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("embedding model ready")

# ChromaDB stores vectors — searches by meaning not exact match
CHROMA_PATH = "/tmp/ptsd_chromadb"
os.makedirs(CHROMA_PATH, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection("ptsd_trials")
except:
    pass

collection = chroma_client.create_collection(
    name="ptsd_trials",
    metadata={"hnsw:space": "cosine"}
)

texts     = chunked_df["chunk_text"].tolist()
ids       = chunked_df["chunk_id"].tolist()
metadatas = [{"nct_id": r["nct_id"], "text_type": r["text_type"]}
             for _, r in chunked_df.iterrows()]

# embed in batches of 100 to avoid memory issues
BATCH = 100
for i in range(0, len(texts), BATCH):
    embeddings = embed_model.encode(texts[i:i+BATCH]).tolist()
    collection.add(
        documents=texts[i:i+BATCH],
        embeddings=embeddings,
        ids=ids[i:i+BATCH],
        metadatas=metadatas[i:i+BATCH]
    )
    if i % 1000 == 0:
        print(f"  {min(i+BATCH, len(texts))}/{len(texts)} embedded")

print(f"gold layer ready — {collection.count()} vectors in ChromaDB")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-bb40173a-b097-4a78-8db2-0d17d7c62099/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


loaded 25384 chunks from silver layer


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model ready
  100/25384 embedded
  1100/25384 embedded
  2100/25384 embedded
  3100/25384 embedded
  4100/25384 embedded
  5100/25384 embedded
  6100/25384 embedded
  7100/25384 embedded
  8100/25384 embedded
  9100/25384 embedded
  10100/25384 embedded
  11100/25384 embedded
  12100/25384 embedded
  13100/25384 embedded
  14100/25384 embedded
  15100/25384 embedded
  16100/25384 embedded
  17100/25384 embedded
  18100/25384 embedded
  19100/25384 embedded
  20100/25384 embedded
  21100/25384 embedded
  22100/25384 embedded
  23100/25384 embedded
  24100/25384 embedded
  25100/25384 embedded
gold layer ready — 25384 vectors in ChromaDB


Setup Groq + define SQL / RAG / Router tools

In [0]:
import os
from groq import Groq

os.environ["GROQ_API_KEY"] = "gsk_xxxxxxxxxxxxxxxxxxxx"
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# if ChromaDB was wiped (serverless resets /tmp), rebuild from silver
try:
    collection = chroma_client.get_collection("ptsd_trials")
    print(f"ChromaDB ready — {collection.count()} vectors")
except:
    print("rebuilding ChromaDB from silver layer...")
    chunked_df = spark.table("ptsd_trials_silver").toPandas()
    collection = chroma_client.create_collection(
        name="ptsd_trials", metadata={"hnsw:space": "cosine"}
    )
    texts     = chunked_df["chunk_text"].tolist()
    ids       = chunked_df["chunk_id"].tolist()
    metadatas = [{"nct_id": r["nct_id"], "text_type": r["text_type"]}
                 for _, r in chunked_df.iterrows()]
    for i in range(0, len(texts), 100):
        embeddings = embed_model.encode(texts[i:i+100]).tolist()
        collection.add(
            documents=texts[i:i+100], embeddings=embeddings,
            ids=ids[i:i+100], metadatas=metadatas[i:i+100]
        )
    print(f"rebuilt — {collection.count()} vectors")

SCHEMA = """
You have access to a Spark SQL table called `ptsd_trials_structured`.

COLUMNS:
- nct_id, title, status, phase, study_type, condition, intervention,
  sponsor, sponsor_type, enrollment (double), start_date, completion_date,
  min_age, max_age, sex

STATUS VALUES (always uppercase):
  RECRUITING, COMPLETED, TERMINATED, WITHDRAWN,
  NOT_YET_RECRUITING, ACTIVE_NOT_RECRUITING,
  ENROLLING_BY_INVITATION, SUSPENDED

WORD MAPPINGS:
- "patients/admitted/participants/people" → enrollment column
- "active/ongoing/open"                  → status = 'RECRUITING'
- "finished/done"                        → status = 'COMPLETED'
- "stopped/failed"                       → status = 'TERMINATED'
- "government"                           → sponsor_type IN ('FED','NIH')

CRITICAL:
- Never filter by condition — every row is already a PTSD trial
- Dates are strings (YYYY-MM-DD) — never CAST to DATE, compare as strings
- Return ONLY the SQL query, no markdown, no explanation
"""

def sql_tool(question, max_retries=3):
    sql = None
    for attempt in range(1, max_retries + 1):
        prompt = question if attempt == 1 else \
                 f"{question}\n\nThis query failed: {sql}\nFix it."
        r = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SCHEMA},
                {"role": "user",   "content": prompt}
            ],
            temperature=0
        )
        sql = r.choices[0].message.content.strip()
        try:
            result_df = spark.sql(sql)
            pdf = result_df.toPandas()
            # send only a summary to Groq, not 800 rows
            if len(pdf) > 10:
                summary = f"Total records: {len(pdf)}\nSample (first 5):\n{pdf.head(5).to_string(index=False)}"
            else:
                summary = pdf.to_string(index=False)
            return summary, sql, result_df
        except Exception as e:
            if attempt == max_retries:
                return f"SQL failed: {e}", sql, None
    return "SQL failed", sql, None

def rag_tool(question, n_results=4):
    # convert question to vector, find closest matching chunks
    embedding = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=embedding, n_results=n_results)
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        if 1 - dist >= 0.4:
            chunks.append({"nct_id": meta["nct_id"],
                           "text_type": meta["text_type"],
                           "similarity": round(1 - dist, 3),
                           "text": doc})

    # if nothing passes threshold, relax and try wider search
    if not chunks:
        results = collection.query(query_embeddings=embedding, n_results=n_results * 2)
        for doc, meta, dist in zip(
            results["documents"][0], results["metadatas"][0], results["distances"][0]
        ):
            chunks.append({"nct_id": meta["nct_id"],
                           "text_type": meta["text_type"],
                           "similarity": round(1 - dist, 3),
                           "text": doc})

    out = ""
    for i, c in enumerate(chunks):
        out += f"\n[{i+1}] {c['nct_id']} ({c['text_type']}, score: {c['similarity']})\n{c['text']}\n"
    return out

def route_question(question):
    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": """Classify the question as:
SQL  — needs numbers, counts, stats, dates, sponsors, phases
RAG  — needs descriptions, eligibility, explanations, trial details
BOTH — needs numbers AND document context together
Return ONLY one word: SQL, RAG, or BOTH"""},
            {"role": "user", "content": question}
        ],
        temperature=0
    )
    return r.choices[0].message.content.strip().upper()

print("all tools ready")

ChromaDB ready — 25384 vectors
all tools ready


Orchestrator: routes question, calls tools, generates answer

In [0]:
def orchestrator(question):
    print(f"\n{'='*60}")
    print(f"question: {question}")

    route = route_question(question)
    print(f"route: {route}")

    sql_summary  = None
    sql_spark_df = None
    rag_results  = None

    if route in ["SQL", "BOTH"]:
        print("running SQL tool...")
        sql_summary, sql, sql_spark_df = sql_tool(question)
        if sql_spark_df:
            display(sql_spark_df)

    if route in ["RAG", "BOTH"]:
        print("running RAG tool...")
        rag_results = rag_tool(question)

    # build context for final LLM call
    context = f"Question: {question}\n\n"
    if sql_summary:
        context += f"Data from SQL:\n{sql_summary}\n\n"
    if rag_results:
        context += f"Relevant trial documents:\n{rag_results}\n\n"
    context += """
Answer clearly and concisely:
- State key numbers directly
- List top 5 trials max if listing
- Cite NCT IDs where relevant
- Keep under 150 words
"""

    print("generating answer...")
    final = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a clinical trials analyst. Be concise and cite trial IDs."},
            {"role": "user",   "content": context}
        ],
        temperature=0.3
    )
    answer = final.choices[0].message.content
    print(f"\nanswer:\n{answer}")

    # offer full table if SQL returned many rows
    if sql_spark_df is not None:
        pdf = sql_spark_df.toPandas()
        if len(pdf) > 5:
            show = input(f"\n{len(pdf)} records found. Show full table? (yes/no): ").strip().lower()
            if show in ["yes", "y"]:
                display(sql_spark_df)

    print(f"{'='*60}\n")

Run it

In [0]:
print("PTSD Clinical Trials Assistant")
print("type 'exit' to quit\n")

while True:
    print("ask a question:")
    question = input(">>> ").strip()

    if not question:
        continue
    if question.lower() in ["exit", "quit", "q"]:
        print("bye")
        break

    orchestrator(question)

PTSD Clinical Trials Assistant
type 'exit' to quit

ask a question:


>>>  

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can